In [ ]:
!pip install transformers rouge_score torch evaluate datasets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 39.2 MB/s eta 0:00:00
   

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import (
    PegasusForConditionalGeneration,
    PegasusTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
from rouge_score import rouge_scorer

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Global Text Contextualizer
class GlobalTextContextualizer(nn.Module):
    def __init__(self, model_dim=768, max_context_length=512):
        super().__init__()
        self.context_embedding = nn.Linear(model_dim, model_dim)
        self.multi_head_attention = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=8
        )
        self.context_fusion = nn.Linear(model_dim * 2, model_dim)
        self.layer_norm = nn.LayerNorm(model_dim)

    def forward(self, input_embeddings, document_context):
        # Ensure consistent dimensions
        if input_embeddings.dim() == 2:
            input_embeddings = input_embeddings.unsqueeze(0)
        if document_context.dim() == 2:
            document_context = document_context.unsqueeze(0)

        # Slice or pad to ensure consistent size
        max_len = min(input_embeddings.size(1), document_context.size(1))
        input_embeddings = input_embeddings[:, :max_len, :]
        document_context = document_context[:, :max_len, :]

        context_embeddings = self.context_embedding(document_context)
        context_attended, _ = self.multi_head_attention(
            input_embeddings,
            context_embeddings,
            context_embeddings
        )

        fused_embeddings = self.context_fusion(
            torch.cat([input_embeddings, context_attended], dim=-1)
        )

        return self.layer_norm(fused_embeddings)

# Load pre-trained model and tokenizer
tokenizer = PegasusTokenizer.from_pretrained('google/pegasus-cnn_dailymail')
base_model = PegasusForConditionalGeneration.from_pretrained('google/pegasus-cnn_dailymail').to(device)

# Initialize Global Text Contextualizer
contextualizer = GlobalTextContextualizer(
    model_dim=base_model.config.d_model
).to(device)

# Read datasets with reduced memory
train_df = pd.read_csv('train_sampled.csv', usecols=['article', 'highlights'])
val_df = pd.read_csv('validation_sampled.csv', usecols=['article', 'highlights'])
test_df = pd.read_csv('test_sampled.csv', usecols=['article', 'highlights'])

# Preprocessing function for batched data
def custom_preprocessing(batch):
    # Tokenize with careful batch handling
    inputs = tokenizer(
        batch['article'],
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)

    labels = tokenizer(
        batch['highlights'],
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)

    # Process in smaller chunks to reduce memory
    with torch.no_grad():
        # Encoder processing
        input_embeddings = base_model.get_encoder()(
            inputs['input_ids']
        ).last_hidden_state

        # Carefully apply contextualizer
        enhanced_embeddings = contextualizer(
            input_embeddings,
            input_embeddings.mean(dim=1, keepdim=True).expand_as(input_embeddings)
        )

    return {
        'input_ids': inputs['input_ids'].cpu(),
        'attention_mask': inputs['attention_mask'].cpu(),
        'labels': labels['input_ids'].cpu(),
    }

# Prepare datasets with preprocessing
train_dataset = Dataset.from_pandas(train_df).map(
    custom_preprocessing,
    batched=True,
    batch_size=4  # Smaller batch size
)

val_dataset = Dataset.from_pandas(val_df).map(
    custom_preprocessing,
    batched=True,
    batch_size=4
)

test_dataset = Dataset.from_pandas(test_df).map(
    custom_preprocessing,
    batched=True,
    batch_size=4
)

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=base_model,
    return_tensors='pt'
)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    gradient_accumulation_steps=8,
    fp16=True,
    predict_with_generate=True,
    logging_steps=50,
    evaluation_strategy="epoch"
)

# Trainer
trainer = Seq2SeqTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

# Training
trainer.train()

# # ROUGE Score Calculation
# scorer = rouge_scorer.RougeScorer(
#     ['rouge1', 'rouge2', 'rougeL'],
#     use_stemmer=True
# )

# # Generate predictions
# predictions, references = [], []
# for item in test_dataset:
#     with torch.no_grad():
#         input_ids = torch.tensor([item['input_ids']]).to(device)
#         generated_ids = base_model.generate(
#             input_ids,
#             max_length=128,
#             num_return_sequences=1
#         )

#     prediction = tokenizer.decode(
#         generated_ids[0],
#         skip_special_tokens=True
#     )
#     reference = tokenizer.decode(
#         item['labels'],
#         skip_special_tokens=True
#     )

#     predictions.append(prediction)
#     references.append(reference)

# # Compute ROUGE scores
# rouge_scores = {
#     'rouge1': {'f1': [], 'precision': []},
#     'rouge2': {'f1': [], 'precision': []},
#     'rougeL': {'f1': [], 'precision': []}
# }

# for pred, ref in zip(predictions, references):
#     scores = scorer.score(ref, pred)
#     rouge_scores['rouge1']['f1'].append(scores['rouge1'].fmeasure)
#     rouge_scores['rouge1']['precision'].append(scores['rouge1'].precision)
#     rouge_scores['rouge2']['f1'].append(scores['rouge2'].fmeasure)
#     rouge_scores['rouge2']['precision'].append(scores['rouge2'].precision)
#     rouge_scores['rougeL']['f1'].append(scores['rougeL'].fmeasure)
#     rouge_scores['rougeL']['precision'].append(scores['rougeL'].precision)

# # Print ROUGE Scores
# print("\nROUGE Scores:")
# for metric, scores in rouge_scores.items():
#     print(f"{metric}:")
#     print(f"  F1-Score: {np.mean(scores['f1']):.4f}")
#     print(f"  Precision: {np.mean(scores['precision']):.4f}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chauhanaditya3112 (chauhanaditya3112-indian-institute-of-technology-kanpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Epoch,Training Loss,Validation Loss
1,5.459500,4.699640
2,0.663300,0.898354
3,0.569600,0.903599
4,0.492100,0.927212


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=1790, training_loss=1.8867701397261807, metrics={'train_runtime': 3779.5156, 'train_samples_per_second': 3.798, 'train_steps_per_second': 0.474, 'total_flos': 2.0682786276900864e+16, 'train_loss': 1.8867701397261807, 'epoch': 4.9864158829676075})

In [ ]:
from rouge_score import rouge_scorer
import numpy as np
import torch

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

# Store predictions and references
predictions, references = [], []

# Iterate over test dataset
for item in test_dataset:
    input_ids = torch.tensor([item["input_ids"]]).to(device)  # Move to GPU/CPU

    # Generate summary
    generated_ids = base_model.generate(
        input_ids,
        max_length=128,
        num_return_sequences=1
    )

    # Decode prediction and reference
    prediction = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    reference = tokenizer.decode(item["labels"], skip_special_tokens=True)

    predictions.append(prediction)
    references.append(reference)

# Store ROUGE scores and precision
rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}
precision_scores = []  # For storing precision values

# Compute ROUGE scores
for pred, ref in zip(predictions, references):
    scores = scorer.score(pred, ref)  # Correct order (prediction first)

    for key in rouge_scores:
        rouge_scores[key].append(scores[key].fmeasure)  # ROUGE scores
        precision_scores.append(scores[key].precision)  # Precision per sample

# Compute final scores
final_scores = {
    "ROUGE-1": np.mean(rouge_scores["rouge1"]),
    "ROUGE-2": np.mean(rouge_scores["rouge2"]),
    "ROUGE-L": np.mean(rouge_scores["rougeL"]),
    "Average Precision": np.mean(precision_scores)  # Compute avg precision
}

# Print results
print("\nModel Evaluation Results:")
for key, value in final_scores.items():
    print(f"{key}: {value:.4f}")


Model Evaluation Results:
ROUGE-1: 0.4506
ROUGE-2: 0.2151
ROUGE-L: 0.3132
Average Precision: 0.3564


In [ ]:
for i in range(5):
  print(predictions[i])
  print(references[i])

Jenny Eclair has developed a yearning to be creative in her spare time . So she decided she wanted to be taught how to master watercolours . So she went on a Painting In Venus break with her other half .
The comedian stayed with Flavours who offer a Painting In Venice break . Jenny and her partner Geof stayed at the farmhouse Villa Bianchi . Days involved sitting in medieval market towns with a brush and prosecco .
The federal government will give Shoshana Hebshi $40,000 as compensation for being humiliated on the 10th anniversary of the 9/11 terrorist attacks . Armed agents forced her from a plane at Detroit Metropolitan Airport, made her undress during a search and held her for hours . Frontier Airlines, the Transportation Security Administration and Wayne County Airport Authority were named in the federal lawsuit . Hebshi was traveling home after visiting a sister in California when was removed from the Frontier Airlines flight after it landed Sept. 11, 2011 . She was seated next to